# Code in Homework 4 for MSDM 5054

Author: LAN, Tianwei 藍天蔚<br>
ID: 21230969<br>
Email: tlanaa@connect.ust.hk

# Problem 2: Classification on 20newsgroup Data

## 1. Random Forest

In [1]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.metrics import confusion_matrix
import warnings
warnings.filterwarnings('ignore')

# Load 100 keywords
with open('20newsgroup/wordlist.txt', 'r') as f:
    wordlist = [line.strip() for line in f.readlines()]
print(f"Loaded {len(wordlist)} keywords.")

# Build Feature Matrix (16242 × 100)
n_posts = 16242
n_keywords = 100
X = np.zeros((n_posts, n_keywords), dtype=int)

with open('20newsgroup/documents.txt', 'r') as f:
    for line in f:
        parts = line.strip().split()
        if len(parts) != 3:
            continue
        post_id = int(parts[0]) - 1
        word_id = int(parts[1]) - 1
        value = int(parts[2])
        X[post_id, word_id] = value
print(f"Feature matrix shape: {X.shape}")

# Load labels from newsgroups
with open('20newsgroup/newsgroups.txt', 'r') as f:
    y = [int(line.strip()) for line in f.readlines()]
y = np.array(y) - 1  # Now labels are 0, 1, 2, 3
class_names = ["comp", "rec", "sci", "talk"]
print(f"Classes: {class_names}")

# Set up Random Forest classifier and hyperparameter grid
rf = RandomForestClassifier(random_state=42, n_jobs=-1)
param_grid = {
    'n_estimators': [300, 350, 400],      # Number of trees in the forest
    'max_features': ['sqrt', 'log2'],     # Number of features to consider at each split
    'min_samples_split': [2, 5, 10],      # Minimum samples required to split an internal node
}

# Perform 5-fold CV grid search
print("Starting hyperparameter tuning via GridSearchCV...")
grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)
grid_search.fit(X, y)

# Return best model and parameters, then calculate error
best_rf = grid_search.best_estimator_
best_params = grid_search.best_params_
print(f"\nBest parameters: {best_params}")

cv_accuracies = cross_val_score(best_rf, X, y, cv=5, scoring='accuracy')
cv_misclassification_error = 1.0 - np.mean(cv_accuracies)
print(f"\n5-Fold CV Mis-classification Error: {cv_misclassification_error:.4f}")

# Train final model on full dataset
best_rf.fit(X, y)
y_pred = best_rf.predict(X)
cm = confusion_matrix(y, y_pred)
print(f"\nConfusion Matrix:\n{cm}")

# Display top 10 most important keywords
importances = best_rf.feature_importances_
top_indices = np.argsort(importances)[::-1][:10]  # Indices of top 10 features

print("\nTop 10 Most Important Keywords:")
for rank, idx in enumerate(top_indices, 1):
    word = wordlist[idx]
    importance_score = importances[idx]
    print(f"{rank:2d}. {word:<15} ({importance_score:.6f})")

Loaded 100 keywords.
Feature matrix shape: (16242, 100)
Classes: ['comp', 'rec', 'sci', 'talk']
Starting hyperparameter tuning via GridSearchCV...
Fitting 5 folds for each of 18 candidates, totalling 90 fits

Best parameters: {'max_features': 'log2', 'min_samples_split': 5, 'n_estimators': 350}

5-Fold CV Mis-classification Error: 0.2364

Confusion Matrix:
[[4349   87   82   87]
 [ 218 3058   88  155]
 [ 331  117 2034  175]
 [ 162  137  110 5052]]

Top 10 Most Important Keywords:
 1. windows         (0.046907)
 2. christian       (0.037066)
 3. car             (0.035491)
 4. god             (0.035284)
 5. government      (0.029720)
 6. team            (0.027125)
 7. jews            (0.023942)
 8. graphics        (0.021183)
 9. religion        (0.020365)
10. space           (0.019041)


## 2. Boosting Tree

In [2]:
import numpy as np
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.metrics import confusion_matrix
import warnings
warnings.filterwarnings('ignore')

# Load 100 keywords from wordlist.txt
with open('20newsgroup/wordlist.txt', 'r') as f:
    wordlist = [line.strip() for line in f.readlines()]
print(f"Loaded {len(wordlist)} keywords.")

# Build binary feature matrix (16242 posts × 100 keywords) from sparse documents.txt
n_posts = 16242
n_keywords = 100
X = np.zeros((n_posts, n_keywords), dtype=int)

with open('20newsgroup/documents.txt', 'r') as f:
    for line in f:
        parts = line.strip().split()
        if len(parts) != 3:
            continue
        post_id = int(parts[0]) - 1
        word_id = int(parts[1]) - 1
        value = int(parts[2])
        X[post_id, word_id] = value
print(f"Feature matrix shape: {X.shape}")

# Load labels from newsgroups.txt (values are 1,2,3,4) and convert to 0-based indexing
with open('20newsgroup/newsgroups.txt', 'r') as f:
    y = [int(line.strip()) for line in f.readlines()]
y = np.array(y) - 1  # Now labels are 0,1,2,3
class_names = ["comp", "rec", "sci", "talk"]
print(f"Classes: {class_names}")

# Initialize Gradient Boosting classifier and define hyperparameter grid
gbm = GradientBoostingClassifier(random_state=42, verbose=0)
param_grid = {
    'n_estimators': [100, 200],      # Number of boosting iterations
    'learning_rate': [0.05, 0.1],    # Shrinkage parameter (controls overfitting)
    'max_depth': [3, 5],             # Depth of each regression tree
    'min_samples_split': [2, 5],     # Minimum samples to split an internal node
}

# Perform 5-fold cross-validated hyperparameter search
print("Starting hyperparameter tuning via GridSearchCV...")
grid_search = GridSearchCV(
    estimator=gbm,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)
grid_search.fit(X, y)

# Retrieve best model and parameters
best_gbm = grid_search.best_estimator_
best_params = grid_search.best_params_
print(f"\nBest parameters: {best_params}")

# Compute 5-fold CV mis-classification error using best model
cv_accuracies = cross_val_score(best_gbm, X, y, cv=5, scoring='accuracy')
cv_misclassification_error = 1.0 - np.mean(cv_accuracies)
print(f"\n5-Fold CV Misclassification Error: {cv_misclassification_error:.4f}")

# Train final model on full dataset and generate confusion matrix
best_gbm.fit(X, y)
y_pred = best_gbm.predict(X)
cm = confusion_matrix(y, y_pred)
print(f"\nConfusion Matrix:\n{cm}")

# Display top 10 most important keywords based on feature importance
importances = best_gbm.feature_importances_
top_indices = np.argsort(importances)[::-1][:10]

print("\nTop 10 Most Important Keywords:")
for rank, idx in enumerate(top_indices, 1):
    word = wordlist[idx]
    importance_score = importances[idx]
    print(f"{rank:2d}. {word:<15} (Importance: {importance_score:.6f})")

Loaded 100 keywords.
Feature matrix shape: (16242, 100)
Classes: ['comp', 'rec', 'sci', 'talk']
Starting hyperparameter tuning via GridSearchCV...
Fitting 5 folds for each of 16 candidates, totalling 80 fits

Best parameters: {'learning_rate': 0.1, 'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 100}

5-Fold CV Misclassification Error: 0.2195

Confusion Matrix:
[[4230   47  155  173]
 [ 269 2773  151  326]
 [ 522   93 1742  300]
 [ 218   80  218 4945]]

Top 10 Most Important Keywords:
 1. windows         (Importance: 0.083766)
 2. god             (Importance: 0.063075)
 3. government      (Importance: 0.052666)
 4. christian       (Importance: 0.048759)
 5. car             (Importance: 0.048355)
 6. team            (Importance: 0.045763)
 7. space           (Importance: 0.030254)
 8. graphics        (Importance: 0.029177)
 9. jews            (Importance: 0.029148)
10. gun             (Importance: 0.026980)


## LDA

In [10]:
import numpy as np
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.metrics import confusion_matrix
import warnings
warnings.filterwarnings('ignore')

# Load 100 keywords
with open('20newsgroup/wordlist.txt', 'r') as f:
    wordlist = [line.strip() for line in f.readlines()]
print(f"Loaded {len(wordlist)} keywords.")

# Build Feature Matrix (16242 × 100)
n_posts = 16242
n_keywords = 100
X = np.zeros((n_posts, n_keywords), dtype=int)

with open('20newsgroup/documents.txt', 'r') as f:
    for line in f:
        parts = line.strip().split()
        if len(parts) != 3:
            continue
        post_id = int(parts[0]) - 1
        word_id = int(parts[1]) - 1
        value = int(parts[2])
        X[post_id, word_id] = value
print(f"Feature matrix shape: {X.shape}")

# Load labels from newsgroups
with open('20newsgroup/newsgroups.txt', 'r') as f:
    y = [int(line.strip()) for line in f.readlines()]
y = np.array(y) - 1  # Now labels are 0, 1, 2, 3
class_names = ["comp", "rec", "sci", "talk"]
print(f"Classes: {class_names}")

# Set up Linear Discriminant Analysis (LDA)
lda = LinearDiscriminantAnalysis()
print("Performing 5-fold cross-validation with LDA...")

# Compute 5-fold CV misclassification error
cv_accuracies = cross_val_score(lda, X, y, cv=5, scoring='accuracy')
cv_misclassification_error = 1.0 - np.mean(cv_accuracies)
print(f"\n5-Fold CV Misclassification Error: {cv_misclassification_error:.4f}")

# Train final model on full dataset
lda.fit(X, y)
y_pred = lda.predict(X)
cm = confusion_matrix(y, y_pred)
print(f"\nConfusion Matrix:\n{cm}")

Loaded 100 keywords.
Feature matrix shape: (16242, 100)
Classes: ['comp', 'rec', 'sci', 'talk']
Performing 5-fold cross-validation with LDA...

5-Fold CV Misclassification Error: 0.2418

Confusion Matrix:
[[4102   45  239  219]
 [ 341 2607  181  390]
 [ 620  119 1530  388]
 [ 268  125  299 4769]]


## 4. QDA

In [11]:
import numpy as np
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.model_selection import cross_val_score
from sklearn.metrics import confusion_matrix
import warnings
warnings.filterwarnings('ignore')

# Load 100 keywords
with open('20newsgroup/wordlist.txt', 'r') as f:
    wordlist = [line.strip() for line in f.readlines()]
print(f"Loaded {len(wordlist)} keywords.")

# Build Feature Matrix (16242 × 100)
n_posts = 16242
n_keywords = 100
X = np.zeros((n_posts, n_keywords), dtype=int)

with open('20newsgroup/documents.txt', 'r') as f:
    for line in f:
        parts = line.strip().split()
        if len(parts) != 3:
            continue
        post_id = int(parts[0]) - 1
        word_id = int(parts[1]) - 1
        value = int(parts[2])
        X[post_id, word_id] = value
print(f"Feature matrix shape: {X.shape}")

# Load labels from newsgroups
with open('20newsgroup/newsgroups.txt', 'r') as f:
    y = [int(line.strip()) for line in f.readlines()]
y = np.array(y) - 1  # Now labels are 0, 1, 2, 3
class_names = ["comp", "rec", "sci", "talk"]
print(f"Classes: {class_names}")

# Set up Quadratic Discriminant Analysis (QDA)
qda = QuadraticDiscriminantAnalysis()
print("Performing 5-fold cross-validation with QDA...")

# Compute 5-fold CV misclassification error
cv_accuracies = cross_val_score(qda, X, y, cv=5, scoring='accuracy')
cv_misclassification_error = 1.0 - np.mean(cv_accuracies)
print(f"\n5-Fold CV Misclassification Error: {cv_misclassification_error:.4f}")

# Train final model on full dataset
qda.fit(X, y)
y_pred = qda.predict(X)
cm = confusion_matrix(y, y_pred)
print(f"\nConfusion Matrix:\n{cm}")


Loaded 100 keywords.
Feature matrix shape: (16242, 100)
Classes: ['comp', 'rec', 'sci', 'talk']
Performing 5-fold cross-validation with QDA...

5-Fold CV Misclassification Error: 0.3169

Confusion Matrix:
[[4444  101   42   18]
 [ 625 2849   16   29]
 [1246  322  911  178]
 [1036 1203   77 3145]]


# Problem 3: Classification on MNIST Data

## 1. SVM with Linear Kernel

In [4]:
import numpy as np
import pandas as pd
from sklearn.svm import LinearSVC
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import time
import warnings
warnings.filterwarnings('ignore')


# Load the training and test datasets
print("Loading datasets...")
train_df = pd.read_csv('MNIST/train_resized.csv')
test_df = pd.read_csv('MNIST/test_resized.csv')


# Filter for only digits 3 and 6
train_filtered = train_df[(train_df['label'] == 3) | (train_df['label'] == 6)]
test_filtered = test_df[(test_df['label'] == 3) | (test_df['label'] == 6)]


# Separate features (pixels) and labels
X_train = train_filtered.iloc[:, 1:].values  # 144 pixel columns
y_train = train_filtered['label'].values     # labels: 3 or 6


X_test = test_filtered.iloc[:, 1:].values    # 144 pixel columns
y_test = test_filtered['label'].values       # labels: 3 or 6


print(f"Training set size: {X_train.shape[0]} samples")
print(f"Test set size: {X_test.shape[0]} samples")


# Define the SVM model with linear kernel
svm = LinearSVC(dual=False, random_state=42)


# Define parameter grid for C (cost)
param_grid = {
    'C': [0.1, 1, 10, 100, 1000]  # Try a range of C values
}


# Perform 5-fold cross-validation to find the best C
print("Starting 5-fold cross-validation for hyperparameter tuning...")
start_time = time.time()
grid_search = GridSearchCV(
    estimator=svm,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)
grid_search.fit(X_train, y_train)
training_time = time.time() - start_time


# Retrieve best model and parameters
best_svm = grid_search.best_estimator_
best_params = grid_search.best_params_
print(f"\nBest parameters: {best_params}")
print(f"Training time: {training_time:.2f} seconds")


# Evaluate on test set
y_pred = best_svm.predict(X_test)
misclassification_error = 1.0 - np.mean(y_pred == y_test)
cm = confusion_matrix(y_test, y_pred)


print(f"\nMisclassification Error on Test Set: {misclassification_error:.4f}")
print(f"\nConfusion Matrix:\n{cm}")


# Optional: Print detailed classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Digit 3', 'Digit 6']))


Loading datasets...
Training set size: 6026 samples
Test set size: 2462 samples
Starting 5-fold cross-validation for hyperparameter tuning...
Fitting 5 folds for each of 5 candidates, totalling 25 fits

Best parameters: {'C': 0.1}
Training time: 0.43 seconds

Misclassification Error on Test Set: 0.0089

Confusion Matrix:
[[1250   12]
 [  10 1190]]

Classification Report:
              precision    recall  f1-score   support

     Digit 3       0.99      0.99      0.99      1262
     Digit 6       0.99      0.99      0.99      1200

    accuracy                           0.99      2462
   macro avg       0.99      0.99      0.99      2462
weighted avg       0.99      0.99      0.99      2462



## 2. SVM with Radial Kernel

In [3]:
import numpy as np
import pandas as pd
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import confusion_matrix, classification_report
import time
import warnings
warnings.filterwarnings('ignore')

# Load the training and test datasets
print("Loading datasets...")
train_df = pd.read_csv('MNIST/train_resized.csv')
test_df = pd.read_csv('MNIST/test_resized.csv')

# Filter for only digits 3 and 6
train_filtered = train_df[(train_df['label'] == 3) | (train_df['label'] == 6)]
test_filtered = test_df[(test_df['label'] == 3) | (test_df['label'] == 6)]

# Separate features (pixels) and labels
X_train = train_filtered.iloc[:, 1:].values
y_train = train_filtered['label'].values

X_test = test_filtered.iloc[:, 1:].values
y_test = test_filtered['label'].values

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set size: {X_train.shape[0]} samples")
print(f"Test set size: {X_test.shape[0]} samples")

# Define the SVM model with radial (RBF) kernel
svm_rbf = SVC(kernel='rbf', random_state=42)

# Define parameter grid for C and gamma
param_grid = {
    'C': [0.1, 1, 10, 100],      # Cost parameter
    'gamma': ['scale', 'auto', 0.001, 0.01, 0.1, 1]  # Kernel coefficient
}

# Perform 5-fold cross-validation to find the best parameters
print("Starting 5-fold cross-validation for hyperparameter tuning (RBF kernel)...")
start_time = time.time()
grid_search_rbf = GridSearchCV(
    estimator=svm_rbf,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)
grid_search_rbf.fit(X_train_scaled, y_train)
training_time_rbf = time.time() - start_time

# Retrieve best model and parameters
best_svm_rbf = grid_search_rbf.best_estimator_
best_params_rbf = grid_search_rbf.best_params_
print(f"\nBest parameters (RBF): {best_params_rbf}")
print(f"Training time (RBF): {training_time_rbf:.2f} seconds")

# Evaluate on test set
y_pred_rbf = best_svm_rbf.predict(X_test_scaled)
misclassification_error_rbf = 1.0 - np.mean(y_pred_rbf == y_test)
cm_rbf = confusion_matrix(y_test, y_pred_rbf)

print(f"\nMisclassification Error on Test Set (RBF): {misclassification_error_rbf:.4f}")
print(f"\nConfusion Matrix (RBF):\n{cm_rbf}")

# Optional: Print detailed classification report
print("\nClassification Report (RBF):")
print(classification_report(y_test, y_pred_rbf, target_names=['Digit 3', 'Digit 6']))

Loading datasets...
Training set size: 6026 samples
Test set size: 2462 samples
Starting 5-fold cross-validation for hyperparameter tuning (RBF kernel)...
Fitting 5 folds for each of 24 candidates, totalling 120 fits

Best parameters (RBF): {'C': 10, 'gamma': 0.001}
Training time (RBF): 16.05 seconds

Misclassification Error on Test Set (RBF): 0.0037

Confusion Matrix (RBF):
[[1256    6]
 [   3 1197]]

Classification Report (RBF):
              precision    recall  f1-score   support

     Digit 3       1.00      1.00      1.00      1262
     Digit 6       1.00      1.00      1.00      1200

    accuracy                           1.00      2462
   macro avg       1.00      1.00      1.00      2462
weighted avg       1.00      1.00      1.00      2462



## 4. SVM Multi-classification on digits {1,2,5,8}

In [ ]:
import numpy as np
import pandas as pd
from sklearn.svm import LinearSVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler
import time
import warnings
warnings.filterwarnings('ignore')

# Load the training and test datasets
print("Loading datasets...")
train_df = pd.read_csv('MNIST/train_resized.csv')
test_df = pd.read_csv('MNIST/test_resized.csv')

# Filter for only digits 1, 2, 5, and 8
train_filtered = train_df[train_df['label'].isin([1, 2, 5, 8])]
test_filtered = test_df[test_df['label'].isin([1, 2, 5, 8])]

# Separate features (pixels) and labels
X_train = train_filtered.iloc[:, 1:].values
y_train = train_filtered['label'].values

X_test = test_filtered.iloc[:, 1:].values
y_test = test_filtered['label'].values

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set size: {X_train.shape[0]} samples")
print(f"Test set size: {X_test.shape[0]} samples")

# Define the SVM model with linear kernel for multi-class classification
svm = LinearSVC(dual=False, random_state=42)

# Define parameter grid for C (cost)
param_grid = {
    'C': [0.1, 1, 10, 100]
}



# Perform 5-fold cross-validation to find the best C
print("Starting 5-fold cross-validation for hyperparameter tuning...")
start_time = time.time()
grid_search = GridSearchCV(
    estimator=svm,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)
grid_search.fit(X_train_scaled, y_train)
training_time = time.time() - start_time

# Retrieve best model and parameters
best_svm = grid_search.best_estimator_
print(f"\nBest parameters: {grid_search.best_params_}")
print(f"Training time: {training_time:.2f} seconds")

# Evaluate on test set
y_pred = best_svm.predict(X_test_scaled)
misclassification_error = 1.0 - np.mean(y_pred == y_test)
cm = confusion_matrix(y_test, y_pred)

print(f"\nMisclassification Error on Test Set: {misclassification_error:.4f}")
print(f"\nConfusion Matrix:\n{cm}")

# Optional: Print detailed classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Digit 1', 'Digit 2', 'Digit 5', 'Digit 8']))

Loading datasets...
Training set size: 11913 samples
Test set size: 4806 samples
Starting 5-fold cross-validation for hyperparameter tuning...
Fitting 5 folds for each of 4 candidates, totalling 20 fits

Best parameters: {'C': 1}
Training time: 52.02 seconds

Misclassification Error on Test Set: 0.0585

Confusion Matrix:
[[1332   12    7   12]
 [  12 1116   23   34]
 [  12   16 1054   44]
 [  26   20   63 1023]]

Classification Report:
              precision    recall  f1-score   support

     Digit 1       0.96      0.98      0.97      1363
     Digit 2       0.96      0.94      0.95      1185
     Digit 5       0.92      0.94      0.93      1126
     Digit 8       0.92      0.90      0.91      1132

    accuracy                           0.94      4806
   macro avg       0.94      0.94      0.94      4806
weighted avg       0.94      0.94      0.94      4806



## 5. SVM all multi-classification

In [5]:
import numpy as np
import pandas as pd
from sklearn.svm import LinearSVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler
import time
import warnings
warnings.filterwarnings('ignore')

# Load the training and test datasets
print("Loading datasets...")
train_df = pd.read_csv('MNIST/train_resized.csv')
test_df = pd.read_csv('MNIST/test_resized.csv')

# Use all 10 classes (no filtering)
X_train = train_df.iloc[:, 1:].values  # All pixel columns
y_train = train_df['label'].values     # All labels: 0–9

X_test = test_df.iloc[:, 1:].values
y_test = test_df['label'].values

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set size: {X_train.shape[0]} samples")
print(f"Test set size: {X_test.shape[0]} samples")
print(f"Number of classes: {len(np.unique(y_train))}")

# Define Linear SVM for multi-class classification
svm = LinearSVC(dual=False, random_state=42)

# Define parameter grid for C
param_grid = {
    'C': [0.1, 1, 10, 100]
}

# Perform 5-fold cross-validation to find the best C
print("Starting 5-fold cross-validation for hyperparameter tuning...")
start_time = time.time()
grid_search = GridSearchCV(
    estimator=svm,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)
grid_search.fit(X_train_scaled, y_train)
training_time = time.time() - start_time

# Retrieve best model and parameters
best_svm = grid_search.best_estimator_
print(f"\nBest parameters: {grid_search.best_params_}")
print(f"Training time: {training_time:.2f} seconds")

# Evaluate on test set
y_pred = best_svm.predict(X_test_scaled)
misclassification_error = 1.0 - np.mean(y_pred == y_test)
cm = confusion_matrix(y_test, y_pred)

print(f"\nMisclassification Error on Test Set: {misclassification_error:.4f}")
print(f"\nConfusion Matrix:\n{cm}")

# Optional: Print detailed classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Loading datasets...
Training set size: 30000 samples
Test set size: 12000 samples
Number of classes: 10
Starting 5-fold cross-validation for hyperparameter tuning...
Fitting 5 folds for each of 4 candidates, totalling 20 fits

Best parameters: {'C': 10}
Training time: 602.36 seconds

Misclassification Error on Test Set: 0.0879

Confusion Matrix:
[[1108    0    1    2    0    8   11    0    7    3]
 [   0 1334    7    5    0    4    1    1   10    1]
 [   9    7 1050   11   23   13   14   20   35    3]
 [   2    9   44 1112    4   32    9   15   16   19]
 [   3    4   11    2 1096    3    7    1    9   39]
 [  14    9    9   46   13  947   24    5   33   26]
 [   9    2   10    2    6   18 1146    0    7    0]
 [   0    3   17    2   12    6    0 1179    1   44]
 [   8   25   17   46    8   33    8    9  958   20]
 [  11    4    5   16   34   10    1   47   10 1015]]

Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.97      0.9